# Training and Testing Dataset

In [ ]:
import pandas as pd
from jupyter_core.migrate import regex
from pandas import DataFrame
from sympy.strategies.core import switch

datasets = [['Original', 'Train', pd.read_csv('../data/duplicate_detect_train.csv')],
            ['Original', 'Test', pd.read_csv('../data/duplicate_detect_test.csv')],
            ['Deduplicate', 'Train', pd.read_csv('../data/unique_detect_train.csv')],
            ['Deduplicate', 'Test', pd.read_csv('../data/unique_detect_test.csv')]]
for index, [distribution, dataset_name, df] in enumerate(datasets):
    df_size = len(df)
    no_size = len(df[df['label'] == 'no'])
    yes_size = len(df[df['label'] == 'yes'])
    project_size = len(df['repository'].unique())

    other_df = datasets[index + 1 if index % 2 == 0 else index - 1][-1]
    other_df_size = len(other_df)
    other_no_size = len(other_df[other_df['label'] == 'no'])
    other_yes_size = len(other_df[other_df['label'] == 'yes'])

    overall_project_size = len(set(df['repository'].unique()).union(set(other_df['repository'].unique())))
    print(
        f"{distribution} & {dataset_name} & {df_size:,} ({df_size / (df_size + other_df_size) * 100:.1f}) & {no_size:,} ({no_size / (no_size + other_no_size) * 100: .1f}) & {yes_size:,} ({yes_size / (yes_size + other_yes_size) * 100:.1f}) & {yes_size / df_size * 100:.1f} & {project_size} ({project_size / overall_project_size * 100:.1f}) \\\\")



# Analysis Utility Methods

In [ ]:
import pandas as pd
from pandas import DataFrame
import math


def bold_max(column_name: str, row_df: DataFrame, df: DataFrame, decimal_digit: int = 2):
    value = row_df[column_name]
    max_value = df[column_name].max()
    if  math.isclose(value, max_value, abs_tol=10**-decimal_digit):
        return "\\textbf{" + f"{value:.{decimal_digit}f}" + "}"
    else:
        return f"{value:.{decimal_digit}f}"


def convert_prompt(prompt):
    if prompt == 'definition':
        return 'No Keyword'
    elif prompt == 'mat':
        return 'MAT'
    elif prompt == 'jitterbug':
        return 'Jitterbug'
    elif prompt == 'gpt':
        return 'GPT 4'
    elif prompt == 'default':
        return 'Ours'
    else:
        return "Unknown"



# non-LLM Result

In [27]:
# 'bert-our_duplicate_code_comments_test': 'MT-BERT',
# 'bert-our_unique_code_comments_test': 'MT-BERT'
citation_map = {
    'Pattern': 'potdar_exploratory_2014',
    'Liu': 'liu_satd_2018',
    'NLP': 'maldonado_using_2017',
    'TM': 'huang_identifying_2018',
    'MAT': 'guo_how_2021',
    'MT-BERT': 'gu_self-admitted_2024'
}
def yes_no_symbol_mapper(val):
    if val == 'yes':
        return r'\checkmark'
    elif val == 'no':
        return r'\xmark'
    else:
        return r'--'
df = pd.read_csv('../cache/output/detect_output_metrics.csv')
model_map = {'pretrained-potdar-Pattern': 'Pattern',
             'pretrained-liu-detector': 'Liu',
             'trained-liu-detector': 'Liu',
             'trained-liu-detector-5fcv': 'Liu',
             'pretrained-TM': 'TM',
             'trained-TM': 'TM',
             'trained-TM-5fcv': 'TM',
             'pretrained-NLP': 'NLP',
             'trained-NLP': 'NLP',
             'trained-NLP-5fcv': 'NLP',
             'pretrained-MAT': 'MAT',
             'pretrained-bert-default': 'MT-BERT',
             'trained-bert-default': 'MT-BERT',
             'trained-bert-5fcv': 'MT-BERT'
             }

MODEL_ORDER = ['Pattern', 'Liu', 'NLP', 'TM', 'MAT', 'MT-BERT']

full_df = df[(df['model'].isin(model_map.keys())) & (df['metric'] == 'yes')].copy()
full_df['model_short'] = full_df['model'].apply(lambda x: model_map[x])
full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
full_df['retrained'] = full_df['model'].apply(lambda x: 'no' if 'pretrained' in x else 'yes')
full_df.loc[full_df['model_short'].isin({'Pattern', 'MAT'}), "retrained"] = '-'
full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})
full_df["cv"] = full_df.apply(
    lambda row: "yes" if row["model"].endswith("5fcv")
    else "-" if row["retrained"] == "-"
    else "no",
    axis=1
)

full_df['retrained_sym'] = full_df['retrained'].apply(yes_no_symbol_mapper)
full_df['cv_sym'] = full_df['cv'].apply(yes_no_symbol_mapper)

for group_index, (group_name, df) in enumerate(sorted(full_df.groupby('dataset'), key=lambda x: x[0], reverse=True)):
    # print("\\hline") if group_index > 0 else None
    print("\\midrule")
    df = df.sort_values(by=['model_order', 'retrained'], ascending=[True, True])
    # assert df.support.min() == df.support.max()
    for loop_index, (index, row) in enumerate(df.iterrows()):
        model_url = row['model']
        # print(f"{len(df)} : {index}")
        is_middle_index = len(df) // 2 == loop_index
        line = ""
        if loop_index == 0:
            line += f"\\multirow{{{len(df)}}}{{*}}{{\\rotatebox[origin=c]{{45}}{{{group_name}}}}}\n"
        line +=f"  & {row['model_short']}~\\cite{{{citation_map[row['model_short']]}}} & {row['retrained_sym']} & {row['cv_sym']}    &   {bold_max('precision', row, df)} & {bold_max('recall', row, df)} & {bold_max('f1-score', row, df)} & {bold_max('mcc', row, df)}"

        line += "\\\\"
        print(line)
print("\\midrule")


\midrule
\multirow{14}{*}{\rotatebox[origin=c]{45}{Original}}
  & Pattern~\cite{potdar_exploratory_2014} & -- & --    &   0.80 & 0.03 & 0.06 & 0.15\\
  & Liu~\cite{liu_satd_2018} & \xmark & \xmark    &   0.38 & 0.58 & 0.46 & 0.46\\
  & Liu~\cite{liu_satd_2018} & \checkmark & \xmark    &   0.30 & 0.57 & 0.40 & 0.41\\
  & Liu~\cite{liu_satd_2018} & \checkmark & \checkmark    &   0.35 & 0.63 & 0.44 & 0.45\\
  & NLP~\cite{maldonado_using_2017} & \xmark & \xmark    &   0.53 & 0.54 & 0.53 & 0.53\\
  & NLP~\cite{maldonado_using_2017} & \checkmark & \xmark    &   0.68 & 0.63 & 0.65 & 0.65\\
  & NLP~\cite{maldonado_using_2017} & \checkmark & \checkmark    &   0.86 & 0.67 & \textbf{0.75} & \textbf{0.75}\\
  & TM~\cite{huang_identifying_2018} & \xmark & \xmark    &   0.09 & 0.56 & 0.15 & 0.20\\
  & TM~\cite{huang_identifying_2018} & \checkmark & \xmark    &   0.10 & 0.86 & 0.17 & 0.26\\
  & TM~\cite{huang_identifying_2018} & \checkmark & \checkmark    &   0.10 & \textbf{0.88} & 0.18 & 0.28\\
  & 

# RQ3 OSS Model

In [ ]:

for target_dataset in  ['duplicate', 'unique']:
    # print(f"\n\n#####################    {target_dataset}  ###################\n\n\n")
    MODEL_ORDER = ['t5-small', 't5-base', 't5-large','t5-xl', 't5-xxl']
    # MODEL_ORDER = ['t5-xl', 't5-xxl']

    df = pd.read_csv('../cache/output/detect_output_metrics.csv')
    full_df = df[(df['model'].str.contains('|'.join(MODEL_ORDER), regex=True)) & (df['metric'] == 'yes')].copy()
    full_df = full_df[full_df['dataset'] == target_dataset]
    full_df['model_short'] = full_df['model'].apply(lambda x: x[x.index('t5-'): x.index('-', x.index('t5-') + 4)])
    full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
    full_df['shot'] = full_df['model'].apply(lambda x: int(x[x.rindex('-', 0, -5) + 1:x.rindex('-')]))
    full_df['prompt'] = full_df['model'].apply(lambda x: x[x.rindex('-', 0, -7) + 1:x.rindex('-', 0, -5)])
    full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})

    full_df = full_df[full_df['shot'] < 6]


    prompt_group = full_df.groupby(['dataset', 'prompt'])
    prompt_counter = 0
    print("\\midrule")
    for prompt_index, ((dataset_name, prompt_name), prompt_group_df) in enumerate(prompt_group):

        if prompt_index > 0:
            print("\\cmidrule(lr){2-15}")
        prompt_index_changed = True
        model_group = sorted(prompt_group_df.groupby(['dataset', 'prompt', 'model_short']), key=lambda x: MODEL_ORDER.index(x[0][-1]))
        for model_group_index, ((_, _, short_model_name), df) in enumerate(model_group):

            df = df.sort_values(by=['dataset', 'model_order', 'shot'], ascending=[False, True, True])
            assert df['support'].min() == df['support'].max()
            shots = 0
            dataset_rows = len(full_df)//len(df)
            line = ""
            if prompt_index == 0 and model_group_index == 0:
                line += f"\\multirow{{{dataset_rows}}}{{*}}{{\\rotatebox[origin=c]{{90}}{{{dataset_name}}}}}"

            if prompt_index_changed:
                if prompt_name == "definition":
                    prompt_tex = f"\\makecell{{{convert_prompt(prompt_name).replace(' ', '\\\\')}}}"
                else:
                    prompt_tex = convert_prompt(prompt_name)
                line += f"&\\multirow{{{len(model_group)}}}{{*}}{{\\centering {prompt_tex}}}"
            else:
                line += "&"

            for loop_index, (index, row) in enumerate(df.iterrows()):
                if loop_index == 0:
                    line += f" & {short_model_name.upper()}"
                model_url = row['model']
                line += f"  & {bold_max('precision', row, full_df)} & {bold_max('recall', row, full_df)} & {bold_max('f1-score', row, full_df)} & {bold_max('mcc', row, full_df)}"



            line += f"  \\\\"
            print(line)
            prompt_index_changed = False


# RQ4 Proprietary Models

In [ ]:
import pandas as pd
for dataset_index, target_dataset in enumerate(['duplicate', 'unique']):
    # print(f"\n\n#####################    {target_dataset}  ###################\n\n\n")
    MODEL_ORDER = ['gemini-2.0-flash', 'gemini-2.5-flash', 'gpt-5-nano', 'gpt-5-mini', 'gpt-5']
    df = pd.read_csv('../cache/output/detect_output_metrics.csv')
    full_df = df[(df['model'].str.contains('|'.join(['gpt-5', 'gemini']), regex=True)) & (df['metric'] == 'yes')].copy()
    full_df = full_df[full_df['dataset'] == target_dataset]
    full_df['model_short'] = full_df['model'].apply(lambda x: x.split('/')[-1][:-7])
    full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
    full_df['shot'] = full_df['model'].apply(lambda x: x[-6:-5])
    full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})

    full_df.head()

    model_group = sorted(full_df.groupby(['dataset', 'model_short']), key=lambda x: x[0], reverse=True)
    for group_index, ((group_name, short_model_name), df) in enumerate(model_group):

        df = df.sort_values(by=['dataset', 'model_order', 'shot'], ascending=[False, True, True])
        assert df['support'].min() == df['support'].max()
        shots = 0
        prompt_group_size =  len(df)

        line = ""
        if group_index > 0 and group_index % 3 == 0: # GPT has 3 models
            print("\\cmidrule(lr){2-2} \\cmidrule(lr){3-14}")
        if group_index == 0:
            line += f"\\multirow{{{(len(model_group))}}}{{*}}{{\\rotatebox[origin=c]{{45}}{{{group_name}}}}}"
        for loop_index, (index, row) in enumerate(df.iterrows()):

            if loop_index == 0:
                line += f" & {short_model_name}"
            model_url = row['model']
            line += f"  & {bold_max('precision', row, full_df)} & {bold_max('recall', row, full_df)} & {bold_max('f1-score', row, full_df)} & {bold_max('mcc', row, full_df)}"



        line += f"  \\\\"
        print(line)

    print("\\midrule")


In [ ]:
import pandas as pd

df = pd.read_csv('../data/duplicate_satd_comment.csv')
df = df[df['label'] == 'requirement']
todo_df = df[df['text'].str.lower().str.startswith('// todo')]
# df.head()
print(len(todo_df) / len(df))


50 repositories ordered by number of comments analyzed

In [ ]:
df = pd.read_csv('../data/repository.csv')
df = df.sort_values(by = "analyzed_comments", ascending = False).head(50)

df = df[["name", "comments", "analyzed_comments", "satd_comments", "percent_satd_comments"]]
df = df.map(lambda x: f"{x:,}" if isinstance(x, int) else x)
df["percent_satd_comments"] = df["percent_satd_comments"].map(lambda x: f"{x:.2f}")
print("projects : ", len(df))
print(df.to_latex(index=False))


50 well-known repositories

In [ ]:
import pandas as pd
sample_repo_ids = [6, 8, 11, 13, 15, 17, 28, 32, 35, 37, 40, 51, 53, 59, 60, 61, 63, 65, 67, 74, 77, 81, 83, 86, 87, 99, 104, 119, 127, 128, 129, 135, 144, 147, 164, 173, 177, 182, 189, 246, 272, 284, 287, 323, 348, 401, 483, 491, 546, 768]

df = pd.read_csv('../data/repository.csv')
df = df[df["id"].isin(sample_repo_ids)]

df = df[["name", "stars", "forks", "watchers", "comments"]]
df = df.map(lambda x: f"{x:,}" if isinstance(x, int) else x)
df = df.sort_values(by = "name", key=lambda s: s.str.lower(), ascending = True)
print("projects : ", len(df))
print(df.to_latex(index=False))




In [ ]:
import pandas as pd
import subprocess
df = pd.read_csv('../data/repository.csv')
x,y = list(map(int, input("Enter project index range : ").split(":")))
urls = df["repo_url"].to_list()
print("Range: {x}-{y}".format(x=x, y=y))
for url in urls[x:y]:
    print(url)
    subprocess.Popen([
        "chromium-browser",
        url
    ])


In [ ]:
import pandas as pd
dp_satd_df = pd.read_csv('../data/duplicate_satd_comment.csv')
stad_map = dict(zip(dp_satd_df["id"], dp_satd_df["label"]))
for name, column in [("classify_n_shot", "label"), ("comment", "type"), ("duplicate_classify_test", "label"), ("duplicate_classify_train", "label"), ("unique_classify_test", "label"), ("unique_classify_train", "label"), ("unique_satd_comment", "label")]:
    file = f'../data/{name}.csv'
    df = pd.read_csv(file)
    df[column] = df["id"].map(lambda id: stad_map[id] if id in stad_map else None)
    df.to_csv(file, index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('../cache/output/detect_output_metrics.csv')
metric_cols = ["precision", "recall", "f1-score", "mcc", "auc", "support"]

# 1. Remove existing aggregated rows (e.g., trained-TM-5fcv)
df_clean = df[~df["model"].str.contains(r"-5fcv$")].copy()

# 2. Select only fold rows (e.g., -5fcv-1 ... -5fcv-5)
df_5fcv = df[df["model"].str.contains(r"-5fcv-\d+$")].copy()

# 3. Extract prefix
df_5fcv["model_prefix"] = df_5fcv["model"].str.replace(r"-5fcv-\d+$", "", regex=True)

# 4. Aggregate
agg_df = (
    df_5fcv
    .groupby(["model_prefix", "dataset", "metric"], as_index=False)
    .agg({
        "precision": "mean",
        "recall": "mean",
        "f1-score": "mean",
        "mcc": "mean",
        "auc": "mean",
        "support": "mean"
    })
)

# 5. Create aggregated model name
agg_df["model"] = agg_df["model_prefix"] + "-5fcv"
agg_df = agg_df.drop(columns=["model_prefix"])

# 6. Reorder columns
agg_df = agg_df[["model", "dataset", "metric"] + metric_cols]

# 7. Append safely
final_df = pd.concat([df_clean, agg_df], ignore_index=True)
final_df.to_csv("../cache/output/detect_output_metrics.csv", index=False)